# SDK DataForge Studio — Colab Practice
This notebook **calls the same functions** as Streamlit. Colab is for experimenting; the published web app runs on Streamlit Community Cloud.

**No ML model training.**

## 1) Upload the complete project ZIP
Upload `SDK_DataForge_Live_Final_Project.zip` from the instructor. The cell extracts it and imports `engine.py`.

In [ ]:
from google.colab import files
from zipfile import ZipFile
from pathlib import Path
import os, sys
uploads = files.upload()  # choose the project ZIP
zip_path = next(name for name in uploads if name.endswith('.zip'))
work = Path('/content/SDK_DataForge_Live_Final')
work.mkdir(exist_ok=True)
with ZipFile(zip_path) as z:
    z.extractall(work)
os.chdir(work)
sys.path.insert(0, str(work))
print('Project files:', [p.name for p in work.iterdir() if p.is_file()])

## 2) Upload the Kaggle CSV
Download the real dataset from [Kaggle](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews), then upload its CSV. If it isn't ready, use the bundled **synthetic** classroom sample for a quick test.

In [ ]:
import io
import pandas as pd
use_real_kaggle = False  # change to True after downloading the Kaggle CSV
if use_real_kaggle:
    dataset_upload = files.upload()
    csv_name = next(name for name in dataset_upload if name.lower().endswith('.csv'))
    df = pd.read_csv(io.BytesIO(dataset_upload[csv_name]))
else:
    df = pd.read_csv('data/sdk_classroom_demo_SYNTHETIC.csv')
print('Shape:', df.shape)
display(df.head())

## 3) EDA and Data Quality

In [ ]:
from engine import quality_report, run_pipeline
print('Missing values:', int(df.isna().sum().sum()))
display(quality_report(df))

## 4) Reusable Pipeline + step-by-step Python Logging

In [ ]:
if 'Age' in df.columns:  # Kaggle schema
    numerical = ['Age', 'Positive Feedback Count']
    categorical = ['Department Name', 'Class Name']
    text_col = 'Review Text'
else:  # Classroom sample
    numerical = ['Review Length']
    categorical = ['Category', 'Channel']
    text_col = 'Review Text'

result = run_pipeline(df, numerical, categorical, text_col)
print('Feature matrix shape:', result['matrix'].shape)
print(result['logs'])

## 5) Text before/after and feature matrix preview

In [ ]:
display(pd.DataFrame({
    'Raw review': df[text_col].head(5),
    'Clean review': result['cleaned'][text_col].head(5)
}))
feature_preview = result['matrix'][:5, :10]
if hasattr(feature_preview, 'toarray'):
    feature_preview = feature_preview.toarray()
display(pd.DataFrame(feature_preview, columns=result['features'][:10]).round(3))

## 6) Exit question
Why should the Streamlit app **import** `run_pipeline()` rather than copy this code into the app? What will happen if we select different columns for numeric, categorical and NLP?